In [1]:
##### Assembles rasters on tonnes and value of livestock products and overall sum of livestock production 
# Distributes FAO country-level production stats to pixels based on livestock density  
# Intermediate step - convert livestock density rasters into absolute livestock rasters

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import shutil

In [2]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# load country data 
production_tonnes = pd.read_csv(f"{cd}/Data/Clean/Production/FAO_data/FAO_production_by_animal.csv")
production_value = pd.read_csv(f"{cd}/Data/Clean/Production/FAO_data/FAO_production_value_clean_commodity.csv")

country_shp = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer='ADM_0')

In [3]:
##### Define function to get total number of livestock in each pixel 

def density_to_absolute (animal_name, animal_code ):
    raster_path = f"{cd}/Data/Raw/FAO_gridded_livestock/GLW4-2020.D-DA.{animal_code}.tif"
    output_path = f"{cd}/Data/Clean/Production/Livestock_count/{animal_name}.tif"

    # load raster
    with rasterio.open(raster_path) as src:
            density = src.read(1).astype(np.float64)   
            nodata  = src.nodata
            meta    = src.meta.copy()
            transform = src.transform
            crs     = src.crs
            height, width = density.shape

            # Aproximate the area of each pixel in m2 using lat and long of each pixel 
            res_deg = abs(transform.e)           # degrees per pixel (latitude)
            res_lon = abs(transform.a)           # degrees per pixel (longitude)
            # latitude of each row centre
            lats = np.array([
                transform.f + (r + 0.5) * transform.e
                for r in range(height)
            ])
            # area = (lon_res * lat_res) * (111.32 km/deg)² * cos(lat)
            pixel_area_km2 = (
                res_lon * res_deg *
                (111.32 ** 2) *
                np.abs(np.cos(np.radians(lats)))
            )[:, np.newaxis]  # shape (height, 1) – broadcast over columns

    if nodata is not None:
        density[density == nodata] = np.nan
    density[density < 0] = np.nan


    #### CALCULATE
    # total animals per pixel = density * area
    # density = animial / km2
    # area = km2

    output = np.full((height, width), np.nan, dtype=np.float64)
    animal_per_pixel = density * pixel_area_km2

    ##### WRITE OUTPUT 

    meta.update(
            dtype   = "float32",
            count   = 1,
            nodata  = -9999,
            compress= "lzw"
        )

    output = animal_per_pixel

    out_arr = output.astype(np.float32)
    out_arr[np.isnan(out_arr)] = -9999

    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(out_arr, 1)

    total_animals = np.nansum(animal_per_pixel)

    return total_animals

In [4]:
def process_absolute_raster(animal_name, animal_code):
    raster_path = f"{cd}/Data/Raw/FAO_gridded_livestock/GLW4-2020.D-DA.{animal_code}.tif"
    output_path = f"{cd}/Data/Clean/Production/Livestock_count/{animal_name}.tif"

    with rasterio.open(raster_path) as src:
        data    = src.read(1).astype(np.float64)
        nodata  = src.nodata
        meta    = src.meta.copy()

    # Mask nodata and invalid values
    if nodata is not None:
        data[data == nodata] = np.nan
    data[data < 0] = np.nan

    # Match output metadata to density_to_absolute outputs
    meta.update(
        dtype   = "float32",
        count   = 1,
        nodata  = -9999,
        compress= "lzw"
    )

    out_arr = data.astype(np.float32)
    out_arr[np.isnan(out_arr)] = -9999

    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(out_arr, 1)

    total_animals = np.nansum(data)

    return total_animals

In [5]:
##### Run gridded to absolute function for animals with density data  

den_names = ['cattle', 'goats', 'sheep', 'pigs', 'buffalo', 'chicken']
den_codes = ['CTL', 'GTS', 'SHP', 'PGS', 'BFL', 'CHK']

for name, code in zip(den_names, den_codes):
    density_to_absolute(name, code)

##### Run cleaning function for animals in absolute already 
den_names = ['horses', 'ducks']
den_codes = ['HOR', 'DCK']

for name, code in zip(den_names, den_codes):
    process_absolute_raster(name, code)

In [6]:
##### Create all raster by adding all animals 

##### Create aggregate raster of total animals across all categories

all_animal_names = ['cattle', 'goats', 'sheep', 'pigs', 'buffalo', 'chicken', 'horses', 'ducks']

aggregate = None
meta = None

for animal_name in all_animal_names:
    raster_path = f"{cd}/Data/Clean/Production/Livestock_count/{animal_name}.tif"
    
    with rasterio.open(raster_path) as src:
        data = src.read(1).astype(np.float64)
        if meta is None:
            meta = src.meta.copy()
    
    # Replace nodata with nan for summation
    data[data == -9999] = np.nan
    
    if aggregate is None:
        aggregate = np.zeros_like(data)
    
    # Add, treating nan as zero (absent data shouldn't void the whole sum)
    aggregate = np.nansum(np.stack([aggregate, data]), axis=0)

##### Write aggregate raster

output_path = f"{cd}/Data/Clean/Production/Livestock_count/all.tif"

meta.update(
    dtype   = "float32",
    count   = 1,
    nodata  = -9999,
    compress= "lzw"
)

out_arr = aggregate.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(output_path, "w", **meta) as dst:
    dst.write(out_arr, 1)


# save copy of all as other for use as proxy
src = f"{cd}/Data/Clean/Production/Livestock_count/all.tif"
dst = f"{cd}/Data/Clean/Production/Livestock_count/other.tif"

shutil.copy(src, dst)

'/Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Production/Livestock_count/other.tif'

In [7]:
##### Define function to distribute country animal production (tonnes) by livestock distirbution 

def apportion_country_production (animal_name):

    ### Define
    raster_path = f"{cd}/Data/Clean/Production/Livestock_count/{animal_name}.tif"
    output_path = f"{cd}/Data/Clean/Production/Livestock_tonnes/{animal_name}_production_tonnes_2020.tif"
    column_title = f"{animal_name}_t"
    country_shp = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer='ADM_0')

    ### Set-up 
    # create dictionary of countries 
    animal_dict = dict(zip(production_tonnes['ISO3'], production_tonnes[column_title]))

    # load raster 
    with rasterio.open(raster_path) as src:
            animals_per_pixel = src.read(1).astype(np.float64)  
            nodata  = src.nodata
            meta    = src.meta.copy()
            transform = src.transform
            crs     = src.crs
            height, width = animals_per_pixel.shape

    if nodata is not None:
        animals_per_pixel[animals_per_pixel == nodata] = np.nan

    # ensure crs's match
    country_shp = country_shp.to_crs(crs)

    # set output shape
    output = np.full((height, width), np.nan, dtype=np.float64)

    total_countries  = len(country_shp)

    # iterate through each country, creating a mask and reapportioning the country total based on livestock 
    for _, row in country_shp.iterrows():
        iso3 = str(row['GID_0']).upper()
        animals = animal_dict.get(iso3, None)

        if animals is None or np.isnan(animals):
            continue   

        # Rasterize country mask
        mask = rasterize(
            [(row.geometry, 1)],
            out_shape=(height, width),
            transform=transform,
            fill=0,
            dtype=np.uint8
        ).astype(bool)

        country_animals = animals_per_pixel.copy()
        country_animals[~mask] = np.nan

        total_animals = np.nansum(country_animals)

        if total_animals <= 0:
            continue  
        else:
            weight = country_animals / total_animals   # fraction per pixel
            output[mask] = weight[mask] * animals

    ### Write output 

    meta.update(
        dtype   = "float32",
        count   = 1,
        nodata  = -9999,
        compress= "lzw"
    )

    out_arr = output.astype(np.float32)
    out_arr[np.isnan(out_arr)] = -9999

    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(out_arr, 1)


In [8]:
##### Run country appotionment function for all animals 

names = ['cattle', 'goats', 'sheep', 'pigs', 'horses', 'ducks', 'buffalo', 'other', 'chicken']

for name in names:
    apportion_country_production(name)

In [9]:
##### Calculate total animal production (tonnes) across all 

animal_names = ['cattle', 'goats', 'sheep', 'pigs', 'horses', 'ducks', 'buffalo', 'other', 'chicken']

# initialise sum array from first raster
with rasterio.open(f"{cd}/Data/Clean/Production/Livestock_tonnes/{animal_names[0]}_production_tonnes_2020.tif") as src:
    total = src.read(1).astype(np.float64)
    nodata = src.nodata
    meta = src.meta.copy()

total[total == nodata] = np.nan

# Track where we've seen at least one valid (non-NaN) value
valid_mask = ~np.isnan(total)
total = np.where(np.isnan(total), 0, total)  # work with 0s internally

# add remaining animals
for animal_name in animal_names[1:]:
    with rasterio.open(f"{cd}/Data/Clean/Production/Livestock_tonnes/{animal_name}_production_tonnes_2020.tif") as src:
        arr = src.read(1).astype(np.float64)
        arr[arr == src.nodata] = np.nan

    this_valid = ~np.isnan(arr)
    valid_mask |= this_valid                      # pixel is valid if ANY layer has data
    total += np.where(np.isnan(arr), 0, arr)      # treat NaN as 0 for summation only

# Restore NaN where NO layer had valid data, then write
total[~valid_mask] = np.nan

# write output
meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")
out_arr = total.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(f"{cd}/Data/Clean/Production/livestock_production_tonnes_2020.tif", "w", **meta) as dst:
    dst.write(out_arr, 1)

In [10]:
#### Clean livestock_value data

# drop livestock commodities 
livestock = ['all', 'cattle', 'chicken', 'goats', 'pigs', 'sheep','horses', 'ducks', 'buffalo']
livestock_value = production_value[production_value['final_group'].isin(livestock)]

# convert to $ 
livestock_value['ag_production_USD_2020'] = livestock_value['ag_production_thousand_USD_2020'] * 1e3

# convert to wide 
livestock_value_wide = livestock_value.pivot_table(index='ISO3', columns='final_group', values='ag_production_USD_2020')
livestock_value_wide = livestock_value_wide.reset_index()

# fill missing with 0's
livestock_value_wide = livestock_value_wide.fillna(0)

/var/folders/48/ky2jtbmj31bfj15cr5gq480w0000gn/T/ipykernel_90321/3284233198.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  livestock_value['ag_production_USD_2020'] = livestock_value['ag_production_thousand_USD_2020'] * 1e3


In [11]:
livestock_value_wide

final_group,ISO3,all,buffalo,cattle,chicken,ducks,goats,horses,pigs,sheep
0,AFG,5.236133e+07,0.0,1.298532e+09,2.866729e+07,0.0,3.345972e+08,0.0,0.000000e+00,7.486958e+08
1,AGO,0.000000e+00,0.0,2.023000e+07,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00
2,ALB,5.785000e+07,0.0,5.806440e+08,2.536870e+08,0.0,8.340300e+07,0.0,1.820400e+07,1.398660e+08
3,ARE,1.905813e+08,0.0,9.068099e+07,1.826384e+08,0.0,3.592019e+08,0.0,0.000000e+00,2.263470e+07
4,ARG,3.119100e+07,0.0,7.121566e+09,3.684462e+09,133000.0,1.785000e+06,90070000.0,2.516000e+07,3.604330e+08
...,...,...,...,...,...,...,...,...,...,...
181,VNM,1.680070e+08,157671000.0,3.293341e+09,5.502939e+09,344652000.0,1.150300e+08,8878000.0,9.841205e+09,0.000000e+00
182,WSM,0.000000e+00,0.0,0.000000e+00,1.233000e+06,0.0,0.000000e+00,0.0,0.000000e+00,0.000000e+00
183,YEM,7.480300e+07,0.0,4.431700e+08,6.160310e+08,0.0,4.441110e+08,0.0,0.000000e+00,3.430040e+08
184,ZAF,1.389040e+08,0.0,3.985114e+09,3.466077e+09,1530000.0,4.331600e+07,2189000.0,4.909960e+08,1.387306e+09


In [12]:
##### Define function to distribute country animal production (value) by livestock distirbution 

def apportion_country_production_value (animal_name, country_shp):

    ### Define
    raster_path = f"{cd}/Data/Clean/Production/Livestock_count/{animal_name}.tif"
    output_path = f"{cd}/Data/Clean/Production/Livestock_value/{animal_name}_production_USD_2020.tif"
    column_title = animal_name

    ### Set-up 
    # create dictionary of countries 
    animal_dict = dict(zip(livestock_value_wide['ISO3'], livestock_value_wide[column_title]))

    # load raster 
    with rasterio.open(raster_path) as src:
            animals_per_pixel = src.read(1).astype(np.float64)  
            nodata  = src.nodata
            meta    = src.meta.copy()
            transform = src.transform
            crs     = src.crs
            height, width = animals_per_pixel.shape

    if nodata is not None:
        animals_per_pixel[animals_per_pixel == nodata] = np.nan

    # ensure crs's match
    country_shp = country_shp.to_crs(crs)

    # set output shape
    output = np.full((height, width), np.nan, dtype=np.float64)

    total_countries  = len(country_shp)

    # iterate through each country, creating a mask and reapportioning the country total based on livestock 
    for _, row in country_shp.iterrows():
        iso3 = str(row['GID_0']).upper()
        animals = animal_dict.get(iso3, None)

        if animals is None or np.isnan(animals):
            continue   

        # Rasterize country mask
        mask = rasterize(
            [(row.geometry, 1)],
            out_shape=(height, width),
            transform=transform,
            fill=0,
            dtype=np.uint8
        ).astype(bool)

        country_animals = animals_per_pixel.copy()
        country_animals[~mask] = np.nan

        total_animals = np.nansum(country_animals)

        if total_animals <= 0:
            continue  
        else:
            weight = country_animals / total_animals   
            output[mask] = weight[mask] * animals

    ### Write output 

    meta.update(
        dtype   = "float32",
        count   = 1,
        nodata  = -9999,
        compress= "lzw"
    )

    out_arr = output.astype(np.float32)
    out_arr[np.isnan(out_arr)] = -9999

    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(out_arr, 1)


In [13]:
##### Run country value appotionment function for all animals 

names = ['cattle', 'goats', 'sheep', 'pigs', 'horses', 'ducks', 'buffalo', 'all', 'chicken']

for name in names:
    apportion_country_production_value(name, country_shp)

In [14]:
##### Calculate total animal production (value) across all 

animal_names = ['cattle', 'goats', 'sheep', 'pigs', 'horses', 'ducks', 'buffalo', 'all', 'chicken']

with rasterio.open(f"{cd}/Data/Clean/Production/Livestock_value/{animal_names[0]}_production_USD_2020.tif") as src:
    total = src.read(1).astype(np.float64)
    nodata = src.nodata
    meta = src.meta.copy()

total[total == nodata] = np.nan

# Track where we've seen at least one valid (non-NaN) value
valid_mask = ~np.isnan(total)
total = np.where(np.isnan(total), 0, total)  # work with 0s internally

for animal_name in animal_names[1:]:
    with rasterio.open(f"{cd}/Data/Clean/Production/Livestock_value/{animal_name}_production_USD_2020.tif") as src:
        arr = src.read(1).astype(np.float64)
        arr[arr == src.nodata] = np.nan

    this_valid = ~np.isnan(arr)
    valid_mask |= this_valid                          # pixel is valid if ANY layer has data
    total += np.where(np.isnan(arr), 0, arr)          # treat NaN as 0 for summation only

# Restore NaN where NO layer had valid data, then write
total[~valid_mask] = np.nan

meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")
out_arr = total.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(f"{cd}/Data/Clean/Production/livestock_production_USD_2020.tif", "w", **meta) as dst:
    dst.write(out_arr, 1)